# phase_2 detector on Kaggle — YOLOv8l, Drive-resumable

Trains the 29-region detector **exactly like the local run**, but checkpoints to
**Google Drive via rclone** so it survives Kaggle's ephemeral storage.

**Why Drive:** `/kaggle/working` is wiped between sessions unless you *Commit*, and
a session can die mid-run. We push the run dir to Drive **every `SYNC_EVERY`
optimizer steps + each epoch**; the next session pulls `last.pt` and `--resume`s.

**Each session:** Settings → Accelerator → **GPU**, then *Run All*. From the 2nd
session on, set `RESUME = 1` in CONFIG. That's the only change.

## CONFIG — edit these

In [ ]:
import os
# ===== code comes from GitHub (cloned in the next cell), not a code dataset =====
PHASE2_SRC = "/kaggle/working/repo/phase_2"
# ===== Kaggle dataset slugs (edit) =====
IMAGES      = "/kaggle/input/datasets/nguynnghin/mimic-cxr-448/mimic-cxr-448"  # 448 images (stem == image_id)
YOLO_LABELS = "/kaggle/input/datasets/nguynnghin/yolo-labels"   # PREBUILT labels (gold 784 ids skipped at link time)

# ===== training =====
MODEL    = "yolov8m.pt"   # m (not l): ~2x faster, near-identical box quality for 29 easy anatomical regions
IMGSZ    = 448            # images are 448
EPOCHS   = 50
FRACTION = 0.5            # train on this fraction of the data (29 regions are "deceptively easy" -> subset is plenty)
RUN_NAME = "det29"
RESUME   = 0             # 0 = fresh; 1 = continue from the Drive checkpoint (2nd session on)

# ===== durable checkpoints (Google Drive via a SERVICE ACCOUNT) =====
# dhint: is defined entirely by the SA + the folder id below -> no rclone.conf, no token.
# Because root = your CHEX-DATA folder, paths DROP the 'CHEX-DATA/' prefix.
DRIVE_FOLDER_ID = "1c6AJ3fjsL449kiMK4xiXfnzwzA4gIo0O"   # CHEX-DATA folder id
RCLONE_REMOTE   = "dhint:phase2_runs"           # = CHEX-DATA/phase2_runs
SYNC_EVERY      = 300                            # push every N optimizer steps

for k, v in dict(PHASE2_SRC=PHASE2_SRC, IMAGES=IMAGES, YOLO_LABELS=YOLO_LABELS, MODEL=MODEL,
                 IMGSZ=IMGSZ, EPOCHS=EPOCHS, FRACTION=FRACTION, RUN_NAME=RUN_NAME, RESUME=RESUME,
                 DRIVE_FOLDER_ID=DRIVE_FOLDER_ID, RCLONE_REMOTE=RCLONE_REMOTE, SYNC_EVERY=SYNC_EVERY).items():
    os.environ[k] = str(v)
print("config set | MODEL =", MODEL, "| FRACTION =", FRACTION, "| RESUME =", RESUME)

In [2]:
# --- get the code from GitHub (instead of uploading a code dataset). Internet ON. ---
!rm -rf /kaggle/working/repo && git clone -q https://github.com/hiennguyendang/phase_2_3_4_5 /kaggle/working/repo
!ls /kaggle/working/repo/phase_2 | head

build_pseudo_scene_graph.py
build_sft_dataset.py
build_yolo_dataset.py
config.py
constants.py
eval_yolo.py
extract_sg_vocab.py
finetune_sg_llm.py
infer_yolo.py
kaggle_sync.py


## 1 — install rclone + auth via a Google **service account**

No OAuth token, no `rclone.conf`. **One-time setup:**
1. **Add-ons → Secrets → Add**: label **`GDRIVE_SA`**, value = the **full JSON** of your
   service-account file (the whole `{...}`).
2. In Google Drive, **share the `CHEX-DATA` folder** (Editor) with the SA's email
   (`client_email` in the JSON, e.g. `kaggle-rclone@…iam.gserviceaccount.com`) — a service
   account has its **own empty Drive**, so it can't see your folder until you share it.
3. Put the `CHEX-DATA` **folder id** (URL `…/folders/<ID>`) into `DRIVE_FOLDER_ID` in CONFIG.

In [3]:
%%bash
set -e
if ! command -v rclone >/dev/null 2>&1; then
  cd /kaggle/working
  curl -sLO https://downloads.rclone.org/rclone-current-linux-amd64.zip
  unzip -q -o rclone-current-linux-amd64.zip
  cp rclone-*-linux-amd64/rclone /usr/local/bin/ && chmod +x /usr/local/bin/rclone
fi
rclone version | head -1

rclone v1.74.3


In [4]:
import os, pathlib
# Define the 'dhint' Google Drive remote ENTIRELY from a service account + folder id, via env vars.
# No rclone.conf, no token. Graceful: if the secret/share/id is missing, training still runs
# WITHOUT Drive sync (checkpoints stay in /kaggle/working).
SYNC_OK = "0"
try:
    from kaggle_secrets import UserSecretsClient
    sa = UserSecretsClient().get_secret("GDRIVE_SA")          # full service-account json
    sa_path = "/kaggle/working/gdrive_sa.json"; pathlib.Path(sa_path).write_text(sa)
    os.environ["RCLONE_CONFIG_DHINT_TYPE"] = "drive"
    os.environ["RCLONE_CONFIG_DHINT_SERVICE_ACCOUNT_FILE"] = sa_path
    os.environ["RCLONE_CONFIG_DHINT_ROOT_FOLDER_ID"] = os.environ["DRIVE_FOLDER_ID"]  # = CHEX-DATA
    if os.system('rclone mkdir "%s"' % os.environ["RCLONE_REMOTE"]) == 0:
        SYNC_OK = "1"; print("remote OK (service account) ->", os.environ["RCLONE_REMOTE"])
        os.system('rclone lsd dhint: 2>/dev/null | head')
    else:
        print("[WARN] remote unreachable -> NO Drive sync. Check: folder shared with the SA email + correct DRIVE_FOLDER_ID")
except Exception as e:
    print("[WARN] GDRIVE_SA secret not set -> NO Drive sync:", e)
os.environ["SYNC_OK"] = SYNC_OK
print("SYNC_OK =", SYNC_OK)

2026/06/26 17:28:56 NOTICE: Config file "/root/.config/rclone/rclone.conf" not found - using defaults


remote OK (service account) -> dhint:phase2_runs
           0 2025-12-04 06:49:35        -1 CheXplus
           0 2025-12-04 17:00:28        -1 Mimic-CXR
           0 2026-06-19 02:33:01        -1 TRACE
           0 2026-05-02 03:57:01        -1 chexplus_processed
           0 2026-06-19 02:33:19        -1 mimic_processed
           0 2026-06-26 17:19:57        -1 phase2_runs
SYNC_OK = 1


## 2 — copy code into the writable dir + install ultralytics

In [5]:
!rm -rf /kaggle/working/phase_2 && cp -r "$PHASE2_SRC" /kaggle/working/phase_2
%cd /kaggle/working/phase_2
!pip install -q ultralytics

/kaggle/working/phase_2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.8 MB/s eta 0:00:00


In [ ]:
# verify GPU BEFORE training (P100 = sm_60 is NOT supported by torch 2.10 -> use T4!)
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "| device_count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0), "| capability:", torch.cuda.get_device_capability(0))
assert torch.cuda.is_available(), "No GPU! Settings -> Accelerator -> GPU T4 x2, then Run All."
assert torch.cuda.get_device_capability(0) >= (7, 0), \
    "GPU too old for torch 2.10 (need sm_70+). P100 is sm_60 -> switch to T4."

## 3 — assemble the YOLO dataset (labels built LOCALLY)

The slow part (open every image for W/H + convert boxes) was done **once locally** with
`build_yolo_dataset.py --labels-only` and uploaded as the **`yolo-labels`** dataset. Here we just
**symlink the matching images** + write `dataset.yaml` — fast, no PIL, no path autodetect headaches.
So you no longer upload the 11 GB scene-graph dataset or the metadata to Kaggle at all.

In [ ]:
import os, glob, zipfile
YL = os.environ["YOLO_LABELS"]
# Kaggle keeps the upload as labels.zip (not auto-extracted) -> unzip once into /kaggle/working
zips = glob.glob(os.path.join(YL, "**", "labels.zip"), recursive=True)
LDIR = YL
if zips:
    LDIR = "/kaggle/working/yolo_labels_ex"
    os.makedirs(LDIR, exist_ok=True)
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(LDIR)
    print("unzipped", zips[0], "->", LDIR)
os.environ["LDIR"] = LDIR
!python link_yolo_images.py --labels-dir "$LDIR" --images-root "$IMAGES" --out /kaggle/working/yolo_ds

In [ ]:
# sanity: dataset.yaml + how many images/labels linked per split (gold 784 skipped from test)
import os, glob
DS = "/kaggle/working/yolo_ds"
print(open(f"{DS}/dataset.yaml").read())
for s in ("train", "val", "test"):
    imgs = glob.glob(f"{DS}/images/{s}/*")
    lbls = glob.glob(f"{DS}/labels/{s}/*")
    print(f"{s}: images={len(imgs)} labels={len(lbls)}")
sample = glob.glob(f"{DS}/images/train/*")[:1]
if sample:
    print("symlink ->", os.path.realpath(sample[0]))

## 4 — train (Drive-synced)

Fresh run pushes checkpoints to Drive every `SYNC_EVERY` steps + each epoch.
If the session dies, set `RESUME=1` in CONFIG and *Run All* again — `train_yolo.py`
pulls `last.pt` from Drive and continues.

In [ ]:
import os
# add --sync-remote only if Drive is configured; --resume only if RESUME=1
# NOTE: do NOT quote the remote — IPython `!...$SYNC` leaks literal quotes into argv,
# which makes rclone treat it as a LOCAL path (silent no-op, nothing reaches Drive).
# The remote has no spaces, so no quotes are needed.
os.environ["SYNC"] = ('--sync-remote %s --sync-every %s' %
                      (os.environ["RCLONE_REMOTE"], os.environ["SYNC_EVERY"])) \
                     if os.environ.get("SYNC_OK") == "1" else ""
os.environ["RESUME_FLAG"] = "--resume" if int(os.environ["RESUME"]) else ""
print("sync:", os.environ["SYNC"] or "(off)", "| resume:", os.environ["RESUME_FLAG"] or "(fresh)")
!python train_yolo.py \
  --data /kaggle/working/yolo_ds/dataset.yaml \
  --model "$MODEL" --device 0 --imgsz $IMGSZ --epochs $EPOCHS --fraction $FRACTION --batch -1 \
  --runs /kaggle/working/runs --name "$RUN_NAME" $SYNC $RESUME_FLAG

## 5 — eval mAP (val; use `--split test` for the gold human-verified set)

In [ ]:
!python eval_yolo.py \
  --weights /kaggle/working/runs/$RUN_NAME/weights/best.pt \
  --data /kaggle/working/yolo_ds/dataset.yaml --split val --device 0

## 6 — grab the trained model

`best.pt` is already on Drive (final push at train end). To pull it anywhere:

```bash
rclone copy dhint:phase2_runs/det29/weights/best.pt ./   # dhint: root = CHEX-DATA
```
Or download it from the Kaggle notebook's **Output** tab (`runs/det29/weights/best.pt`).

In [ ]:
import os
RUN = os.environ["RUN_NAME"]
if os.environ.get("SYNC_OK") == "1":
    os.system('rclone copy /kaggle/working/runs/%s "%s/%s" --quiet' % (RUN, os.environ["RCLONE_REMOTE"], RUN))
    print("pushed final run -> %s/%s" % (os.environ["RCLONE_REMOTE"], RUN))
else:
    print("Drive sync off -> download from the notebook 'Output' tab: runs/%s/weights/best.pt" % RUN)
!ls -lh /kaggle/working/runs/$RUN_NAME/weights/